# Fortgeschrittene Rekursionsmuster in UnifyWeaver

Dieses Notebook demonstriert die vier wichtigsten Rekursionsmuster, die UnifyWeaver erkennen und optimieren kann:

1. **Endrekursion (Tail Recursion)** - Iterative Schleifen mit Akkumulatoren
2. **Lineare Rekursion (Linear Recursion)** - Einzelner rekursiver Aufruf mit Memoisierung
3. **Baumrekursion (Tree Recursion)** - Mehrere rekursive Aufrufe auf Strukturteilen
4. **Wechselseitige Rekursion (Mutual Recursion)** - Prädikate, die sich zyklisch gegenseitig aufrufen

## Lernziele

- Verschiedene Rekursionsmuster verstehen
- Sehen, wie UnifyWeaver jedes Muster erkennt und optimiert
- Leistungsmerkmale vergleichen
- Lernen, wann welches Muster verwendet werden sollte

## Einrichtung

Initialisieren der UnifyWeaver-Umgebung.

In [ ]:
% Load initialization
['../init'].

% Load necessary modules
use_module(unifyweaver(core/recursive_compiler)).
use_module(unifyweaver(core/advanced/pattern_matchers)).

## Muster 1: Endrekursion (Tail Recursion)

Endrekursion verwendet einen Akkumulator zur Übergabe von Zwischenergebnissen, und der rekursive Aufruf ist die **letzte Aktion** in der Funktion.

### Beispiel: Elemente in einer Liste zählen

In [ ]:
% Define tail-recursive count_items
:- dynamic count_items/3.

% Base case: empty list, return accumulator
count_items([], Acc, Acc).

% Recursive case: increment accumulator, recurse on tail
count_items([_|T], Acc, N) :-
    Acc1 is Acc + 1,
    count_items(T, Acc1, N).  % ← Tail position!

### In Prolog testen

In [ ]:
% Test: count items in [a,b,c,d,e]
\+ \+ (
    count_items([a,b,c,d,e], 0, _N),
    format('Count: ~w~n', [_N])
).

### Mustererkennung prüfen

In [ ]:
% Check if detected as tail recursive
\+ \+ (
    is_tail_recursive_accumulator(count_items/3, _AccInfo),
    format('Tail recursive: ~w~n', [_AccInfo])
).

### Nach Bash kompilieren

In [ ]:
% Compile and save
\+ \+ (
    compile_recursive(count_items/3, [], _BashCode),
    setup_call_cleanup(
        open('../output/count_items_demo.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('✓ Compiled count_items to Bash with tail recursion optimization')
).

### Generiertes Bash testen

In [ ]:
%%bash
source ../output/count_items_demo.sh
echo "Counting items in [a,b,c,d,e]:"
count_items "[a,b,c,d,e]" 0 ""

## Muster 2: Lineare Rekursion (Linear Recursion)

Lineare Rekursion besitzt **genau einen** rekursiven Aufruf pro Klausel, wobei Berechnungen nach der Rückkehr des rekursiven Aufrufs stattfinden.

### Beispiel: Fakultät (Factorial)

In [ ]:
% Define factorial
:- dynamic factorial/2.

% Base case
factorial(0, 1).

% Recursive case: exactly ONE recursive call
factorial(N, F) :-
    N > 0,
    N1 is N - 1,
    factorial(N1, F1),  % ← One recursive call
    F is N * F1.        % ← Computation after call

### In Prolog testen

In [ ]:
% Test: factorial of 5
\+ \+ (
    factorial(5, _F),
    format('5! = ~w~n', [_F])
).

### Mustererkennung prüfen

In [ ]:
% Check if detected as linear recursive
is_linear_recursive_streamable(factorial/2),
writeln('✓ Detected as linear recursion').

### Nach Bash kompilieren

In [ ]:
% Compile and save
\+ \+ (
    compile_recursive(factorial/2, [], _BashCode),
    % Keep function definitions only; Brush treats sourced scripts as direct execution
    split_string(_BashCode, "\n", "\r", _BashLines),
    append(_LibraryLines, ["# Auto-execute when run directly (not when sourced)"|_], _BashLines),
    atomics_to_string(_LibraryLines, "\n", _LibraryCode),
    setup_call_cleanup(
        open('../output/factorial_demo.sh', write, _Stream),
        write(_Stream, _LibraryCode),
        close(_Stream)),
    writeln('✓ Compiled factorial to Bash with fold-based linear recursion')
).

### Generiertes Bash testen

In [ ]:
%%bash
source ../output/factorial_demo.sh
echo "Factorial of 5:"
factorial 5 ""
echo ""
echo "Factorial of 10:"
factorial 10 ""

## Muster 3: Baumrekursion (Tree Recursion)

Baumrekursion führt **mehrere** rekursive Aufrufe aus, um verschiedene Teile einer Struktur zu verarbeiten.

### Beispiel: Baumsumme (Tree Sum)

In [ ]:
% Define tree_sum for binary trees
% Tree format: [Value, LeftSubtree, RightSubtree] or []
:- dynamic tree_sum/2.

% Base case: empty tree has sum 0
tree_sum([], 0).

% Recursive case: sum = value + left_sum + right_sum
tree_sum([V, L, R], Sum) :-
    tree_sum(L, LS),   % ← First recursive call
    tree_sum(R, RS),   % ← Second recursive call
    Sum is V + LS + RS.

### In Prolog testen

In [ ]:
% Test: tree_sum of [5, [3, [1, [], []], []], [2, [], []]]
%       5
%      / \
%     3   2
%    /
%   1
\+ \+ (
    tree_sum([5, [3, [1, [], []], []], [2, [], []]], _Sum),
    format('Tree sum: ~w (expected 11)~n', [_Sum])
).

### Nach Bash kompilieren

In [ ]:
% Compile and save
\+ \+ (
    compile_recursive(tree_sum/2, [], _BashCode),
    setup_call_cleanup(
        open('../output/tree_sum_demo.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('✓ Compiled tree_sum to Bash with tree recursion')
).

### Generiertes Bash testen

In [ ]:
%%bash
source ../output/tree_sum_demo.sh
echo "Tree sum of [5,[3,[1,[],[]],[]],[2,[],[]]]:"
tree_sum "[5,[3,[1,[],[]],[]],[2,[],[]]]"

## Muster 4: Wechselseitige Rekursion (Mutual Recursion)

Wechselseitige Rekursion tritt auf, wenn zwei oder mehr Prädikate sich gegenseitig in einem Zyklus aufrufen.

### Beispiel: Gerade (Even) und Ungerade (Odd)

In [ ]:
% Define mutually recursive is_even and is_odd
:- dynamic is_even/1.
:- dynamic is_odd/1.

% is_even base case
is_even(0).

% is_even recursive: N is even if N-1 is odd
is_even(N) :-
    N > 0,
    N1 is N - 1,
    is_odd(N1).  % ← Calls is_odd

% is_odd base case
is_odd(1).

% is_odd recursive: N is odd if N-1 is even
is_odd(N) :-
    N > 1,
    N1 is N - 1,
    is_even(N1).  % ← Calls is_even

### In Prolog testen

In [ ]:
% Test even/odd
is_even(0), writeln('✓ 0 is even').
is_even(4), writeln('✓ 4 is even').
is_odd(3), writeln('✓ 3 is odd').
is_odd(7), writeln('✓ 7 is odd').

### Auf wechselseitige Rekursion prüfen

In [ ]:
% Build call graph and find SCCs
\+ \+ (
    use_module(unifyweaver(core/advanced/call_graph)),
    use_module(unifyweaver(core/advanced/scc_detection)),

    build_call_graph([is_even/1, is_odd/1], _Graph),
    format('Call graph: ~w~n', [_Graph]),

    find_sccs(_Graph, _SCCs),
    format('SCCs (mutual recursion groups): ~w~n', [_SCCs])
).

### Nach Bash kompilieren

In [ ]:
% Compile the mutual recursion group
\+ \+ (
    use_module(unifyweaver(core/advanced/mutual_recursion)),

    compile_mutual_recursion([is_even/1, is_odd/1], [], _BashCode),
    split_string(_BashCode, "\n", "\r", _BashLines),
    append(_LibraryLines, ["# Main dispatch: route command line calls to functions"|_], _BashLines),
    atomics_to_string(_LibraryLines, "\n", _LibraryCode),
    setup_call_cleanup(
        open('../output/even_odd_demo.sh', write, _Stream),
        write(_Stream, _LibraryCode),
        close(_Stream)),
    writeln('✓ Compiled is_even/is_odd to Bash with shared memoization')
).

### Generiertes Bash testen

In [ ]:
%%bash
source ../output/even_odd_demo.sh
echo "Testing is_even and is_odd:"
is_even 0 >/dev/null && echo "✓ 0 is even"
is_even 4 >/dev/null && echo "✓ 4 is even"
is_odd 3 >/dev/null && echo "✓ 3 is odd"
is_odd 7 >/dev/null && echo "✓ 7 is odd"
is_even 5 >/dev/null 2>&1 || echo "✓ 5 is not even"

## Mustervergleich

Vergleichen wir die Eigenschaften der einzelnen Muster:

| Muster | Rekursive Aufrufe | Optimierung | Platzkomplexität | Am besten geeignet für |
|:--------|:----------------|:-------------|:-----------------|:---------|
| **Endrekursion** | 1 (in Endposition) | Iterative Schleife | O(1) | Akkumulatoren, lineare Scans |
| **Lineare Rekursion** | 1 (beliebige Position) | Fold + Memoisierung | O(n) Memo-Tabelle | Fibonacci, Fakultät |
| **Baumrekursion** | 2+ (Strukturteile) | Strukturelle Zerlegung | O(Tiefe) Stack | Baum-/Graphoperationen |
| **Wechselseitig** | 1+ (prädikatsübergreifend) | Geteilte Memoisierung | O(n) geteilte Tabelle | Gerade/Ungerade, gegenseitige Definitionen |

## Reihenfolge der Mustererkennung

UnifyWeaver versucht den Musterabgleich in dieser Reihenfolge:

1. **Endrekursion** (am effizientesten)
2. **Lineare Rekursion** (sofern nicht verboten)
3. **Baumrekursion** (strukturell)
4. **Wechselseitige Rekursion** (SCC-Erkennung)
5. **Basis-Rekursion** (Fallback)

Sie können die Erkennung mit `forbid_linear_recursion/1` beeinflussen.

## Übung: Jetzt sind Sie dran!

Versuchen Sie, diese Prädikate zu definieren und zu kompilieren:

### 1. Endrekursive Summe
```prolog
sum_list([], Acc, Acc).
sum_list([H|T], Acc, Sum) :-
    Acc1 is Acc + H,
    sum_list(T, Acc1, Sum).
```

### 2. Linear-rekursive Fibonacci-Folge
```prolog
fib(0, 0).
fib(1, 1).
fib(N, F) :-
    N > 1,
    N1 is N - 1,
    N2 is N - 2,
    fib(N1, F1),
    fib(N2, F2),
    F is F1 + F2.
```

### 3. Baumhöhe
```prolog
tree_height([], 0).
tree_height([_, L, R], H) :-
    tree_height(L, HL),
    tree_height(R, HR),
    H is max(HL, HR) + 1.
```

In [ ]:
% Your code here!


## Zusammenfassung

In diesem Notebook haben Sie gelernt:

✅ Die vier wichtigsten Rekursionsmuster in UnifyWeaver

✅ Wie man jedes Muster in Prolog definiert

✅ Wie UnifyWeaver jedes Muster erkennt und optimiert

✅ Die Leistungsmerkmale jedes Musters

✅ Wann welches Muster verwendet werden sollte

## Nächste Schritte

Fahren Sie mit **Notebook 3: Visualisierung des Aufruf-Graphen** fort, um mehr über fortgeschrittene Code-Analyse und Visualisierung zu erfahren!